In [4]:
"""
Phase 2 - Commit 1
Loads the raw F1 dataset, constructs the binary podium target, 
applies minimal median imputation to numeric columns only, and produces an 
isolated raw train/test split for the Task 1 unmodified-eLCS floor run. 
No leakage remedition happens here - that will be done in later stages.
"""

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from skeLCS import eLCS

RAW_DATA_PATH = "f1_raw_25299773.csv"

# Load the raw F1 dataset

df_raw = pd.read_csv(RAW_DATA_PATH, na_values="\\N", low_memory=False)

df_raw["podium"] = df_raw["positionOrder"].between(1, 3).astype(int) # Setting the target variable to 1 if the driver finished in the top 3, otherwise 0

# Minimal column selection to avoid leakage and reduce dimensionality. Exclude columns that are identifiers or directly related to the target variable.

excluded_cols = {
    "position", "positionText", "positionOrder",
    "resultId", "driverId", "raceId", "constructorId", "circuitId",
    "number", "number_driver", "number_quali",
    "podium",
}

numeric_cols = df_raw.select_dtypes(include=[np.number]).columns
feature_cols_raw = [c for c in numeric_cols if c not in excluded_cols]

X_raw = df_raw[feature_cols_raw].copy()
y_raw = df_raw["podium"].copy()

# Minimal missing value imputation for numeric columns only, using median imputation.

X_raw = X_raw.fillna(X_raw.median(numeric_only=True))

print("Raw feature matrix shape:", X_raw.shape)
print("Podium class balance:\n", y_raw.value_counts(normalize=True).round(4))

# Isolated 80/20 train/test split for the Task 1 unmodified-eLCS floor run. No leakage remediation is applied here.

X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    X_raw, y_raw,
    test_size=0.2,
    stratify=y_raw,
    random_state=42,
)

print("X_train_raw:", X_train_raw.shape, "| X_test_raw:", X_test_raw.shape)




"""
Phase 2 - Commit 2
Trains the unmodified eLCS model, with default hyperparameters, on the
raw/minimally-processed dataset. Establishes the Task 2
floor performance benchmark before any cleaning, feature engineering,
or hyperparameter tuning, which happens in later commits.
"""

import time
import numpy as np
from sklearn.metrics import classification_report, roc_auc_score
from skeLCS import eLCS

elcs_baseline = eLCS(learning_iterations=1000, random_state=42) # Initialize the eLCS model with 1000 learning iterations and a fixed random state for reproducibility

print("Baseline eLCS parameters:", elcs_baseline.get_params()) # Print the default hyperparameters of the eLCS model
print("Raw training matrix shape:", X_train_raw.shape) # Print the shape of the training feature matrix

# Train the eLCS model on the raw training data and measure the time taken for training 
X_train_raw_arr = X_train_raw.values
y_train_raw_arr = y_train_raw.values
X_test_raw_arr = X_test_raw.values
y_test_raw_arr = y_test_raw.values

# Measure the time taken to fit the eLCS model on the training data
fit_start = time.time()
elcs_baseline.fit(X_train_raw_arr, y_train_raw_arr)
fit_runtime_seconds = time.time() - fit_start 

print(f"Baseline eLCS training runtime: {fit_runtime_seconds:.2f} seconds")  # Print the time taken to train the model

y_pred_raw = elcs_baseline.predict(X_test_raw_arr) # Predict the target variable for the test set using the trained eLCS model

print("\nClassification report (raw floor):")
# Print the classification report showing precision, recall, f1-score, and support for each class
print(classification_report(y_test_raw_arr, y_pred_raw, target_names=["non-podium", "podium"])) 

# Calculate the ROC-AUC score for the predictions made by the eLCS model on the test set
roc_auc_floor = roc_auc_score(y_test_raw_arr, y_pred_raw)
print(f"ROC-AUC (raw floor): {roc_auc_floor:.4f}") 




Raw feature matrix shape: (26759, 24)
Podium class balance:
 podium
0    0.8731
1    0.1269
Name: proportion, dtype: float64
X_train_raw: (21407, 24) | X_test_raw: (5352, 24)
Baseline eLCS parameters: {'N': 1000, 'acc_sub': 0.99, 'beta': 0.2, 'chi': 0.8, 'delta': 0.1, 'discrete_attribute_limit': 10, 'do_GA_subsumption': True, 'do_correct_set_subsumption': False, 'fitness_reduction': 0.1, 'init_fit': 0.01, 'learning_iterations': 1000, 'match_for_missingness': False, 'mu': 0.04, 'nu': 5, 'p_spec': 0.5, 'random_state': 42, 'reboot_filename': None, 'selection_method': 'tournament', 'specified_attributes': array([], dtype=float64), 'theta_GA': 25, 'theta_del': 20, 'theta_sel': 0.5, 'theta_sub': 20, 'track_accuracy_while_fit': False}
Raw training matrix shape: (21407, 24)
Baseline eLCS training runtime: 0.81 seconds

Classification report (raw floor):
              precision    recall  f1-score   support

  non-podium       0.88      1.00      0.94      4673
      podium       0.92      0.08

In [ ]:
"""
Phase 2 - Commit 3
Builds the baseline shared CSV file for the team, with leakage-stripped features 
and minimal median imputation applied to numeric columns only.
Parses messy alphanumeric qualifying string timestamps into total float seconds,
handles historical temporal features, drops meaningless un-ordered identity arrays,
and exports the team's shared unscaled matrix as 'f1_preprocessed_base.csv'.
"""

import numpy as np
import pandas as pd

RAW_DATA_PATH = "f1_raw_25299773.csv" # Path to the raw dataset
SHARED_OUTPUT_PATH = "f1_preprocessed_base.csv" # Path to save the preprocessed dataset

df_engineered = pd.read_csv(RAW_DATA_PATH, na_values="\\N", low_memory=False) # Load the raw dataset into a DataFrame

# Create a binary target variable 'podium' indicating whether the driver finished in the top 3
df_engineered["podium"] = df_engineered["positionOrder"].between(1, 3).astype(int) 


leakage_cols_to_drop = [
    "points", "milliseconds", "time", "position", "positionText",
    "positionOrder", "fastestLap", "fastestLapTime", "fastestLapSpeed",
    "rank", "statusId",
] # Columns that are directly related to the target variable or contain information that would not be available at prediction time, and thus should be dropped to prevent data leakage.
df_engineered = df_engineered.drop(columns=leakage_cols_to_drop)

def parse_quali_string_to_seconds(value): # Parses the qualitative string timestamp into total float seconds
    if pd.isna(value):
        return 0.0
    try:
        minutes_part, seconds_part = str(value).split(":") # Split the string into minutes and seconds parts
        calculated_seconds = int(minutes_part) * 60 + float(seconds_part) # Convert the minutes part to seconds and add it to the seconds part to get the total time in seconds
    except (ValueError, AttributeError): 
        return 0.0
    
    if calculated_seconds > 180.0: # If the calculated seconds exceed 180, return 0.0 to handle potential outliers or incorrect data
        return 0.0
    return calculated_seconds

for target_col in ["q1", "q2", "q3"]:
    df_engineered[target_col] = df_engineered[target_col].apply(parse_quali_string_to_seconds) # Apply the parsing function to the qualifying time columns to convert them into total float seconds

df_engineered["dob"] = pd.to_datetime(df_engineered["dob"], errors="coerce")
df_engineered["date"] = pd.to_datetime(df_engineered["date"], errors="coerce")
df_engineered["age_at_race"] = (df_engineered["date"] - df_engineered["dob"]).dt.days / 365.25

text_descriptor_cols = df_engineered.select_dtypes(exclude=[np.number]).columns.tolist() # Identify all non-numeric columns in the DataFrame, which are likely to be text descriptors or categorical variables that may not be suitable for certain machine learning models without further preprocessing.
df_engineered = df_engineered.drop(columns=text_descriptor_cols)

identity_code_cols = [
    "resultId", "driverId", "raceId", "constructorId", "circuitId",
    "qualifyId", "constructorId_quali", "driverStandingsId",
    "constructorStandingsId", "number", "number_driver", "number_quali",
]
df_engineered = df_engineered.drop(columns=[col for col in identity_code_cols if col in df_engineered.columns]) # Drop identity code columns that are present in the DataFrame to prevent potential data leakage and reduce dimensionality.

feature_cols_clean = [col for col in df_engineered.columns if col != "podium"]
df_engineered[feature_cols_clean] = df_engineered[feature_cols_clean].fillna(
    df_engineered[feature_cols_clean].median(numeric_only=True) 
) # Fill missing values in the feature columns with the median of each column, calculated only for numeric columns

print("Cleaned Feature Shape:", df_engineered.shape) # Print the shape of the cleaned DataFrame after preprocessing
print("Target Class Skew Balance:\n", df_engineered["podium"].value_counts(normalize=True).round(4)) 

df_engineered.to_csv(SHARED_OUTPUT_PATH, index=False) # Save the cleaned and preprocessed DataFrame to a CSV file for team sharing
print(f"Saved shared team preprocessing foundation to: {SHARED_OUTPUT_PATH}")


Cleaned Feature Shape: (26759, 19)
Target Class Skew Balance:
 podium
0    0.8731
1    0.1269
Name: proportion, dtype: float64
Saved shared team preprocessing foundation to: f1_preprocessed_base.csv
